## Pipeline

```
📷  Smartphone Photo
        │
        ▼
┌───────────────────┐    ┌──────────────────────────────┐
│  SEGMENTER        │    │  WHAT IT DOES                │
│  SAM 2            │───▶│  Isolates the dish region     │
│  (GrabCut here)   │    │  Returns: binary mask (H×W)   │
└───────────────────┘    └──────────────────────────────┘
        │
        ▼
┌───────────────────┐    ┌──────────────────────────────┐
│  DEPTH ESTIMATOR  │    │  WHAT IT DOES                │
│  ml-depth-pro     │───▶│  Estimates distance per pixel │
│  (classical here) │    │  Returns: depth map (H×W) [m] │
└───────────────────┘    └──────────────────────────────┘
        │
        ▼
┌───────────────────┐    ┌──────────────────────────────┐
│  VOLUME ESTIMATOR │    │  WHAT IT DOES                │
│  RANSAC + integral│───▶│  Fits a reference plane       │
│  (same algorithm) │    │  Integrates height × area     │
└───────────────────┘    └──────────────────────────────┘
        │
        ▼
   Volume in ml
   Portion label
```

## Conceptual Map


1. **Segmentation** — *Where is the dish?*  (2D spatial problem)
2. **Depth estimation** — *How far away is each pixel?* (ill-posed inverse problem)
3. **Embedding** — *What dish is this?* (semantic similarity problem)
4. **Volume** — *How much is there?* (3D geometry problem)

Separate model so each can be improved independently or swapped

## Key Concepts by Notebook

### 01 — Segmenter
- **Energy minimisation** — GrabCut minimises an energy function combining data fit and spatial smoothness
- **Gaussian Mixture Model** — models the colour distribution of foreground and background
- **Min-cut / Max-flow** — the discrete optimisation step that relabels pixels
- **Watershed** — topographic flooding that places boundaries on colour ridges

### 02 — Depth Estimator
- **Monocular depth ambiguity** — why a single image cannot uniquely determine depth without priors
- **Defocus blur** — Laplacian variance as a sharpness/depth proxy
- **Dark channel prior** — haze as a signal for depth
- **Perspective geometry** — back-projecting pixels to 3D using the pinhole model

### 03 — Embedder
- **HOG** — histogram of gradient orientations as a shape/texture descriptor
- **Cosine similarity** — why L2-normalised dot products equal cosine similarity
- **Transformer architecture** — patch embedding, CLS token, multi-head attention, positional encoding
- **Self-supervised learning** — how DINOv2 learns without labels

### 04 — Volume Estimator
- **RANSAC** — robust plane fitting that ignores outliers (food drips, noise)
- **Backprojection** — pinhole model to convert (u, v, Z) → (X, Y, Z)
- **Discrete integration** — summing thin columns: V = Σ h_i × A_i
- **Pixel area** — how the real-world size of a pixel depends on depth Z